# 🎯 [PRC07_M07] 추천된 대체 부품의 규격 적합성 검토 및 조달 승인을 위한 HITL 워크플로우 설계
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/PRC07/prc07_m07_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 조달·계약·국산화 (Level 3) |
| **미션 ID** | PRC07_M07 |
| **미션 주제** | 추천된 대체 부품의 규격 적합성 검토 및 조달 승인을 위한 HITL 워크플로우 설계 |
| **소요 시간** | 약 3시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
무기체계 부품은 해외 의존도가 높고 공급선이 하나뿐이거나 단종(DMSMS) 공고가 나오면 조달이 막히기 쉬운데, 위험 신호를 사람이 문서와 전화로 모으면 국산화 후보를 찾기까지 수개월이 걸린다. 본 프로젝트는 부품 공급망 자료에서 조달 병목과 위험 신호를 찾고 국산화 대체 후보를 자동으로 추천하는 도구를 만드는 과제로, 업무 단계별 병목과 위험 신호 기준, 추천 수정안, 계약 의뢰 근거와 변경 이력, 업무 효과 측정까지 한 흐름으로 구현한다. 교육용 가상 부품 100종과 가상 국가·가상 업체 자료로 공급망 데이터 스키마 정의, 공급업체 자료 정제와 단종 공고 파싱, 룰 기반 조달 위험 스크리닝, 텍스트 유사도와 지식 그래프를 쓴 대체 부품 추천, 붕괴 위험도와 조기경보, 위험 지도 대시보드와 오류 처리, 규격 검토와 승인 HITL 워크플로우, 공급망 차단 스트레스 시험, AAR 가중치 재보정, 통합 패키지까지 단계별로 만든다. 최종 목표는 위험 부품을 조기에 찾아 국산화 후보를 근거와 함께 제시하되, 규격 적합성 판단과 조달 승인은 담당자가 하도록 지원하는 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

> **💡 [오프라인 폐쇄망 및 로컬 주피터 환경 안내]**
> - 외부 인터넷망(Colab) 외에 군 인트라넷 또는 로컬 환경에서 실행할 경우, 아래 명령어로 필요한 표준 라이브러리를 사전 설치하십시오:
>   `pip install numpy pandas matplotlib scipy scikit-learn`
> - 인터넷이 제한된 환경의 경우 사전 번들링된 Wheel(`.whl`) 패키지 디렉토리를 통해 설치할 수 있습니다:
>   `pip install --no-index --find-links=./wheels/ -r requirements.txt`

> 📁 **[안내: 프로젝트 표준 산출물 디렉터리 가이드]**
> - 본 프로젝트는 단순한 일회성 코드 실습을 넘어 실제 부대에 배포 가능한 업무 도구(Tool)를 완성하는 과정입니다.
> - 실습 중 작성하는 결과물은 역할에 따라 아래 표준 디렉터리에 자동으로 저장·누적됩니다:
>   - `docs/`: 조달 프로세스 분석서, 스키마 정의서, UAT 보고서, 사용자 매뉴얼 등 행정·기술 문서
>   - `src/`: 데이터 로더, 16종 위험 스크리닝 엔진, AI 추천 모델, 대시보드 UI, HITL 결재 FSM 등 핵심 실행 모듈
>   - `dist/`: 폐쇄망 및 타 부서 배포를 위한 원클릭 설치 스크립트(`setup.sh`) 및 최종 패키지

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")

# ============================================================================
# [산출물 자동 저장 유틸리티] 프로젝트 표준 디렉토리(docs/, src/, dist/) 자동 연계
# ============================================================================
def save_deliverable(filepath: str, content: str):
    """프로젝트 표준 deliverables 디렉토리에 산출물 파일을 자동 생성합니다."""
    dirpath = os.path.dirname(filepath)
    if dirpath:
        os.makedirs(dirpath, exist_ok=True)
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(content.strip() + '\n')
    print(f"📦 [산출물 저장 완료] {filepath}")


---

## 🎯 미션 07. 추천된 대체 부품의 규격 적합성 검토 및 조달 승인을 위한 HITL 워크플로우 설계
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

추천된 대체 부품의 규격 적합성 검토 및 조달 승인 HITL 워크플로우를 설계한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **추천된 대체 부품의 규격 적합성 검토와 조달 승인을 위한 HITL 워크플로우를 설계하고 이력 무결성과 권한 통제를 검증한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **시스템 제안, 조달담당 상신, 기술검토관 규격 검토, 보안담당 검토, 승인권자 결재의 순서를 상태 기계로 구현하고 역할별 권한을 최소로 둔다 과정을 진행하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/workflow/prc07_approval_system.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군 보안 규정 및 행정 절차 부합성

#### 미션 PASS 조건

- 정량 기준: 이력 기록 무결성 100%, 승인 권한 우회 시도 차단율 100%


- 진행 기준: 결재선 승인/반려 시뮬레이션 및 이력 조회 화면 구현


- 검증 방법: 권한별 접근 제어(RBAC) 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""위험 상위 부품의 대체 부품 추천 결과를 조달담당 상신 -> 기술검토관 규격 검토 -> 보안담당 검토 -> 승인권자 결재의 HITL 워크플로우로 처리하되, 시스템의 자동 확정과 권한 우회 시도를 차단하고 모든 행위를 해시 체인 감사 로그에 남겨 이력 무결성과 위변조 탐지를 검증한다."""
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 무기체계 부품·공급망 데이터 생성기 (실제 체계·업체·국가·단가 정보 없음)
# 부품 100종(해외 조달 약 80%)의 공급망 속성과, 국내 부품 카탈로그(과거 승인된 국산 대체품 +
# 설명은 비슷하지만 규격이 맞지 않는 유사 후보)를 함께 만든다.
# ---------------------------------------------------------------------------
COUNTRIES = ["가상국가A", "가상국가B", "가상국가C", "국내"]
CATS = {
    "전원모듈": dict(words=["DC-DC", "컨버터", "절연형", "전원", "정출력"], syn={"컨버터": "변환장치", "절연형": "아이솔레이션", "정출력": "안정화"}, std="MIL-STD-704", iface="단자대"),
    "커넥터": dict(words=["원형", "커넥터", "방수형", "다핀", "나사체결"], syn={"커넥터": "접속자", "방수형": "밀폐형", "나사체결": "스크루결합"}, std="MIL-DTL-38999", iface="원형핀"),
    "유압밸브": dict(words=["유압", "솔레노이드", "밸브", "방향제어", "4포트"], syn={"밸브": "개폐장치", "방향제어": "유로전환", "솔레노이드": "전자식"}, std="SAE-AS5440", iface="플랜지"),
    "베어링": dict(words=["볼", "베어링", "고속", "밀봉형", "회전축"], syn={"베어링": "축받이", "밀봉형": "실드형", "고속": "고회전"}, std="ISO-492", iface="축결합"),
    "광학렌즈": dict(words=["적외선", "렌즈", "게르마늄", "반사방지", "집광"], syn={"렌즈": "광학창", "집광": "초점형성", "반사방지": "AR코팅"}, std="MIL-PRF-13830", iface="마운트"),
    "자이로센서": dict(words=["광섬유", "자이로", "각속도", "센서", "관성"], syn={"센서": "감지기", "각속도": "회전율", "관성": "INS용"}, std="IEEE-952", iface="직렬통신"),
    "RF증폭기": dict(words=["RF", "전력증폭기", "광대역", "모듈", "송신"], syn={"전력증폭기": "파워앰프", "송신": "출력단", "광대역": "와이드밴드"}, std="MIL-STD-461", iface="SMA"),
    "제어보드": dict(words=["FPGA", "제어보드", "임베디드", "연산", "입출력"], syn={"제어보드": "컨트롤카드", "연산": "신호처리", "임베디드": "내장형"}, std="VITA-46", iface="백플레인"),
}
V_OPTS, W_OPTS = [12, 24, 28], [25, 50, 100]
VENDORS = ["가상정밀(주)", "가상전자(주)", "가상유공압(주)", "가상광학(주)", "가상테크(주)", "가상시스템(주)"]


def describe(cat, words, s):
    return f"{cat} {' '.join(words)} {s['v']}V {s['w']}W {s['tmin']}~{s['tmax']}C {s['std']}"


def _sig(x):
    return 1 / (1 + np.exp(-x))


def make_parts(n=100, seed=7):
    """부품 목록(공급망 속성 포함)과 국내 카탈로그를 만든다. 반환: (parts DataFrame, catalog DataFrame)"""
    rng = np.random.default_rng(seed)
    cats, rows, cat_rows = list(CATS), [], []
    for i in range(n):
        cat = cats[i % len(cats)]; c = CATS[cat]
        spec = {"v": int(rng.choice(V_OPTS)), "w": int(rng.choice(W_OPTS)), "tmin": int(rng.choice([-55, -40])),
                "tmax": int(rng.choice([85, 125])), "size_mm": float(round(rng.uniform(20, 120), 1)), "std": c["std"]}
        words = [str(x) for x in rng.permutation(c["words"])[:4]]
        country = str(rng.choice(COUNTRIES, p=[0.35, 0.3, 0.15, 0.2]))
        overseas = country != "국내"
        n_sup = int(rng.choice([1, 2, 3], p=[0.45, 0.35, 0.2])) if overseas else int(rng.choice([1, 2, 3]))
        lead = float(round(rng.gamma(3, 45 if overseas else 25) + 20))
        eol = str(rng.choice(["정상", "단종예고", "단종"], p=[0.7, 0.2, 0.1]))
        crit = str(rng.choice(["A", "B", "C"], p=[0.3, 0.4, 0.3]))
        z = (-2.6 + 1.1 * (n_sup == 1) + 0.9 * overseas + 0.006 * (lead - 100) + 1.2 * (eol == "단종예고")
             + 2.0 * (eol == "단종") + rng.normal(0, 0.6))
        p_late = float(np.clip(_sig(z) * 0.8 + 0.05, 0, 0.9))
        late = int(rng.binomial(12, p_late))
        demand = int(rng.integers(20, 400))
        rows.append({"part_no": f"P-{i + 1:04d}", "category": cat, "desc": describe(cat, words, spec), **spec,
                     "iface": c["iface"], "country": country, "overseas": overseas, "n_suppliers": n_sup,
                     "lead_time_days": lead, "orders_12m": 12, "late_orders": late,
                     "avg_delay_days": float(round(late and rng.gamma(2, 12 + 20 * p_late), 1)),
                     "eol_status": eol, "ltb_months": int(rng.integers(1, 25)) if eol == "단종예고" else None,
                     "price_change_pct": float(round(rng.normal(4 + 12 * p_late, 8), 1)), "criticality": crit,
                     "annual_demand": demand, "stock_qty": int(rng.integers(0, demand)),
                     "defect_rate": float(round(abs(rng.normal(0.8 + 2 * p_late, 0.8)), 2)),
                     "export_license": bool(overseas and rng.random() < 0.35),
                     "disrupted_next12m": bool(rng.random() < _sig(z + 0.4))})
        if not overseas:
            continue
        # 과거 승인된 국산 대체품(정답): 동의어로 서술, 규격 호환(전력 여유 허용, 일부는 크기 차이가 커 장착 검토 필요)
        eq_words = [c["syn"].get(w, w) for w in words]
        hard = rng.random() < 0.15                                   # 서술이 많이 다른 사례
        if hard:
            eq_words = [c["syn"].get(w, w) for w in c["words"] if w not in words][:1] + eq_words[1:3]
        eq = {**spec, "w": spec["w"] if rng.random() < 0.7 else W_OPTS[min(W_OPTS.index(spec["w"]) + 1, 2)],
              "size_mm": float(round(spec["size_mm"] * (rng.uniform(1.11, 1.15) if rng.random() < 0.12 else rng.uniform(0.95, 1.05)), 1))}
        cat_rows.append({"cand_no": f"D-{len(cat_rows) + 1:04d}", "category": cat, "desc": describe(cat, eq_words, eq), **eq,
                         "iface": c["iface"], "vendor": str(rng.choice(VENDORS)), "approved_for": f"P-{i + 1:04d}", "kind": "equivalent"})
        # 유사 오답 후보: 설명은 거의 같지만 전압이 다르거나(5V/48V) 상용 온도 등급(0~70C)
        for kind in ("volt_mismatch", "temp_mismatch"):
            d = dict(spec)
            if kind == "volt_mismatch":
                d["v"] = int(rng.choice([5, 48]))                           # 체계에서 쓰지 않는 전압
            else:
                d["tmin"], d["tmax"] = 0, 70
            d["size_mm"] = float(round(spec["size_mm"] * rng.uniform(0.95, 1.05), 1))
            cat_rows.append({"cand_no": f"D-{len(cat_rows) + 1:04d}", "category": cat, "desc": describe(cat, words, d), **d,
                             "iface": c["iface"], "vendor": str(rng.choice(VENDORS)), "approved_for": None, "kind": kind})
    parts = pd.DataFrame(rows)
    catalog = pd.DataFrame(cat_rows).sample(frac=1, random_state=seed).reset_index(drop=True)
    return parts, catalog
# ---------------------------------------------------------------------------

# ---- 대체 부품 추천 공통 함수: 텍스트 유사도 + 규격 적합성 + 지식 그래프 이웃 유사도 ----
WEIGHTS = {"text": 0.35, "spec": 0.45, "graph": 0.20}


def spec_fit(p, c):
    """규격 적합성 0~1: 6개 조건 충족 비율. 전압·온도 한계(필수 조건)를 못 맞추면 점수를 절반으로 깎는다."""
    conds = [p["v"] == c["v"], c["w"] >= p["w"], c["tmin"] <= p["tmin"], c["tmax"] >= p["tmax"],
             abs(c["size_mm"] - p["size_mm"]) / p["size_mm"] <= 0.10, p["iface"] == c["iface"]]
    s = sum(conds) / len(conds)
    return (s * 0.5 if not (conds[0] and conds[2] and conds[3]) else s), conds


def graph_nodes(row):
    """지식 그래프에서 부품 노드와 연결된 이웃 노드 집합(범주·표준·인터페이스·기능어·등급)."""
    words = set(row["desc"].split()[1:-4])
    grade = "군용등급" if row["tmin"] <= -40 else "상용등급"
    return {f"cat:{row['category']}", f"std:{row['std']}", f"if:{row['iface']}", f"grade:{grade}", f"v:{row['v']}"} | {f"w:{w}" for w in words}


def build_graph(catalog):
    """동의어 사전을 그래프 간선(기능어 = 동의어)으로 넣어 서술이 달라도 같은 기능 노드로 연결되게 한다."""
    syn = {f"w:{b}": f"w:{a}" for c in CATS.values() for a, b in c["syn"].items()}
    canon = lambda nodes: {syn.get(x, x) for x in nodes}
    return {"canon": canon, "nodes": [canon(graph_nodes(r)) for _, r in catalog.iterrows()]}


def graph_sim(g, pn, cn):
    """두 부품이 공유하는 이웃 노드 비율(Jaccard)."""
    pn = g["canon"](pn)
    return len(pn & cn) / max(len(pn | cn), 1)


def score_table(parts_df, catalog):
    """해외 부품 x 같은 범주 국내 후보 쌍마다 텍스트·규격·그래프 점수를 계산한 표(가중치 적용 전)."""
    vec = TfidfVectorizer(token_pattern=r"[^ ]+").fit(list(parts_df["desc"]) + list(catalog["desc"]))
    text = cosine_similarity(vec.transform(parts_df["desc"]), vec.transform(catalog["desc"]))
    g = build_graph(catalog)
    rows = []
    for i, (_, p) in enumerate(parts_df.iterrows()):
        pn = graph_nodes(p)
        for j, (_, c) in enumerate(catalog.iterrows()):
            if c["category"] != p["category"]:
                continue
            sf, conds = spec_fit(p, c)
            rows.append({"part_no": p["part_no"], "cand_no": c["cand_no"], "vendor": c["vendor"], "text": text[i, j],
                         "spec": sf, "graph": graph_sim(g, pn, g["nodes"][j]),
                         "unmet": ",".join(n for n, ok in zip(["전압", "전력", "저온", "고온", "크기", "인터페이스"], conds) if not ok) or "-",
                         "kind": c["kind"], "correct": c["approved_for"] == p["part_no"]})
    return pd.DataFrame(rows)


def recommend(table, weights=None, top=3):
    """가중합 점수로 부품별 상위 top개 후보를 고른다."""
    w = weights or WEIGHTS
    t = table.assign(score=w["text"] * table["text"] + w["spec"] * table["spec"] + w["graph"] * table["graph"])
    return t.sort_values(["part_no", "score"], ascending=[True, False]).groupby("part_no").head(top)


def hit_rate(recs, k=1):
    """과거 승인 이력(정답 대체품)이 상위 k개 안에 들어간 부품 비율."""
    return recs.groupby("part_no").apply(lambda d: d["correct"].head(k).any(), include_groups=False).mean()

# ---- 역할과 권한(최소 권한 원칙): 시스템은 '제안'만 할 수 있고 확정 권한이 없다 ----
PERM = {"시스템": {"propose"}, "조달담당": {"submit", "view"}, "기술검토관": {"review_spec", "view"}, "보안담당": {"review_sec", "view"},
        "승인권자": {"approve", "view"}, "감사관": {"view", "audit"},
        "부서장": {"submit", "approve", "view"}}            # 소규모 부서는 상신과 결재를 겸할 수 있으나 같은 건은 분리
NEXT = {"제안": ("submit", "상신"), "상신": ("review_spec", "규격검토"), "규격검토": ("review_sec", "보안검토"), "보안검토": ("approve", "승인")}
USERS = {"sys": "시스템", "kim": "조달담당", "lee": "기술검토관", "park": "보안담당", "choi": "승인권자", "jung": "감사관", "oh": "부서장"}
FLAGGED_VENDORS = {"가상시스템(주)"}           # [교육용가상자료] 보안 검토에서 추가 확인이 필요한 업체 표기


class Chain:
    """해시 체인 감사 로그: 각 기록이 직전 기록의 해시를 포함해 중간 수정이 드러난다."""

    def __init__(self):
        self.log = []

    def add(self, **body):
        prev = self.log[-1]["hash"] if self.log else "GENESIS"
        body = {"seq": len(self.log), **body}
        self.log.append({**body, "prev": prev, "hash": hashlib.sha256((prev + json.dumps(body, sort_keys=True, ensure_ascii=False)).encode()).hexdigest()})

    def verify(self):
        prev = "GENESIS"
        for e in self.log:
            b = {k: v for k, v in e.items() if k not in ("prev", "hash")}
            if e["prev"] != prev or hashlib.sha256((prev + json.dumps(b, sort_keys=True, ensure_ascii=False)).encode()).hexdigest() != e["hash"]:
                return e["seq"]
            prev = e["hash"]
        return None


class Workflow:
    def __init__(self):
        self.audit, self.cases, self.blocked = Chain(), {}, []

    def _deny(self, who, case, action, why):
        self.audit.add(event="DENIED", who=who, case=case, action=action, reason=why)
        self.blocked.append((who, case, action, why)); return False, why

    def act(self, who, case, action, reason="", evidence=""):
        role = USERS.get(who)
        if role is None:
            return self._deny(who, case, action, "미등록 사용자")
        if action == "delete_log":
            return self._deny(who, case, action, "감사 로그 삭제·수정 권한은 누구에게도 없음")
        if action not in PERM[role]:
            return self._deny(who, case, action, f"권한 없음({role} 역할에 {action} 권한 없음)")
        if action == "propose":
            self.cases[case] = {"state": "제안", "submitter": None}
            self.audit.add(event="PROPOSE", who=who, case=case, reason=reason); return True, "제안 등록"
        c = self.cases.get(case)
        if c is None:
            return self._deny(who, case, action, "존재하지 않는 건")
        if c["state"] in ("승인", "반려"):
            return self._deny(who, case, action, f"이미 종결된 건({c['state']})")
        need, nxt = NEXT[c["state"]]
        if action != need:
            return self._deny(who, case, action, f"단계 위반(현재 {c['state']}, 필요 행위 {need})")
        if not reason.strip() or not evidence.strip():
            return self._deny(who, case, action, "사유·근거 입력 필수")
        if action == "submit":
            c["submitter"] = who
        if action == "approve" and who == c["submitter"]:
            return self._deny(who, case, action, "상신자와 승인자 분리 원칙 위반")
        c["state"] = nxt
        self.audit.add(event=action.upper(), who=who, role=role, case=case, to=nxt, reason=reason, evidence=evidence); return True, nxt

    def reject(self, who, case, reason, evidence):
        """반려: 검토자·승인권자가 어느 단계에서든 사유·근거와 함께 할 수 있다."""
        c = self.cases[case]
        role = USERS.get(who)
        if role not in ("승인권자", "기술검토관", "보안담당", "부서장") or c["state"] in ("승인", "반려"):
            return self._deny(who, case, "reject", "반려 권한 없음 또는 종결된 건")
        if not reason.strip() or not evidence.strip():
            return self._deny(who, case, "reject", "사유·근거 입력 필수")
        c["state"] = "반려"
        self.audit.add(event="REJECTED", who=who, role=role, case=case, reason=reason, evidence=evidence); return True, "반려"


# ---- 상신 대상: 위험 상위 해외 부품과 추천 1순위 후보 ----
parts, catalog = make_parts(100, seed=7)
over = parts[parts["overseas"]].reset_index(drop=True)
table = score_table(over, catalog)
top = recommend(table).groupby("part_no").head(1).set_index("part_no")
risk = (30 * (over["n_suppliers"] == 1) + 25 * (over["late_orders"] / 12) + 20 * (over["eol_status"] != "정상") + 10 * over["criticality"].eq("A")).to_numpy()
picked = over.assign(risk=risk).sort_values("risk", ascending=False).head(12)
rng = np.random.default_rng(4)

wf = Workflow()
rows = []
for k, (_, p) in enumerate(picked.iterrows(), 1):
    case, r = f"C{k:02d}", top.loc[p["part_no"]]
    wf.act("sys", case, "propose", reason=f"{p['part_no']} 위험 점수 {p['risk']:.0f}, 추천 {r['cand_no']}")
    wf.act("kim", case, "submit", "단종·단일 공급선 위험 부품 대체 검토 요청", f"위험 점수 {p['risk']:.0f}점, 후보 점수 {r['score']:.2f}")
    hard = [u for u in r["unmet"].split(",") if u in ("전압", "저온", "고온")]
    if rng.random() < 0.15:
        rows.append((case, p["part_no"], r["cand_no"], "무응답 -> 보류", "자동 확정 없음: 기술검토관 응답 대기")); continue
    if hard:
        wf.reject("lee", case, f"필수 규격 미충족({','.join(hard)})", f"규격 적합성 {r['spec']:.2f}")
        rows.append((case, p["part_no"], r["cand_no"], wf.cases[case]["state"], f"규격 반려: {','.join(hard)}")); continue
    wf.act("lee", case, "review_spec", "필수 규격(전압·온도) 충족 확인" + (f", 미충족 {r['unmet']}는 시험 조건부" if r["unmet"] != "-" else ""), f"규격 적합성 {r['spec']:.2f}")
    if r["vendor"] in FLAGGED_VENDORS:
        wf.reject("park", case, "업체 보안 확인 미완료", f"업체 {r['vendor']} 추가 확인 필요")
        rows.append((case, p["part_no"], r["cand_no"], wf.cases[case]["state"], "보안 반려: 업체 확인 필요")); continue
    wf.act("park", case, "review_sec", "업체·공급 경로 보안 확인", f"업체 {r['vendor']} 확인 완료(가상)")
    ok = r["score"] >= 0.80 and p["risk"] >= 40
    if ok:
        wf.act("choi", case, "approve", "승인: 위험 부품 국산화 시험 착수", f"후보 점수 {r['score']:.2f}, 위험 {p['risk']:.0f}")
    else:
        wf.reject("choi", case, "반려: 추천 점수 또는 위험도 기준 미달(추가 시험 후 재상신)", f"후보 점수 {r['score']:.2f}, 위험 {p['risk']:.0f}")
    rows.append((case, p["part_no"], r["cand_no"], wf.cases[case]["state"], "승인권자 결정" if ok else "점수 미달 반려"))
print("[결재 결과] 12건")
print(pd.DataFrame(rows, columns=["건", "부품", "후보", "최종 상태", "비고"]).to_string(index=False))
st = pd.Series([r[3] for r in rows]).value_counts().to_dict()
print("상태 집계:", st, "(모의 결재자 기준, 실제 결재는 사람이 한다)")

# ---- 권한 우회·절차 위반 시도 ----
wf.act("sys", "T01", "propose", reason="시험 건")
wf.act("kim", "T01", "submit", "시험 상신", "근거")
attempts = [
    ("시스템 자동 승인", lambda: wf.act("sys", "T01", "approve", "자동", "자동")),
    ("조달담당이 규격 검토 대행", lambda: wf.act("kim", "T01", "review_spec", "대행", "근거")),
    ("검토 단계 건너뛰고 승인", lambda: wf.act("choi", "T01", "approve", "바로 승인", "근거")),
    ("미등록 사용자 접근", lambda: wf.act("guest", "T01", "review_spec", "열람", "근거")),
    ("사유 없는 검토", lambda: wf.act("lee", "T01", "review_spec", "", "")),
    ("감사 로그 삭제", lambda: wf.act("kim", "T01", "delete_log", "정리", "근거")),
    ("감사관이 승인 시도", lambda: wf.act("jung", "T01", "approve", "승인", "근거")),
    ("종결된 건 재승인", lambda: wf.act("choi", "C01", "approve", "재승인", "근거")),
]
wf.act("sys", "T02", "propose", reason="시험"); wf.act("oh", "T02", "submit", "상신", "근거")
wf.act("lee", "T02", "review_spec", "규격 확인", "적합성 1.00"); wf.act("park", "T02", "review_sec", "보안 확인", "확인 완료")
attempts.append(("상신자가 본인 건 승인(분리 원칙)", lambda: wf.act("oh", "T02", "approve", "본인 승인", "근거")))
print("\n[권한 우회·절차 위반 시도]")
blocked = 0
for name, fn in attempts:
    ok, why = fn()
    blocked += (not ok)
    print(f"  {name:22s}-> {'차단' if not ok else '통과(결함)'}: {why}")
print(f"우회·위반 시도 차단 {blocked}/{len(attempts)} ({blocked / len(attempts):.0%}) (목표 100%: {'달성' if blocked == len(attempts) else '미달'})")

# ---- 이력 무결성 ----
bad = wf.audit.verify()
n_cases_logged = sum(1 for e in wf.audit.log if e["event"] == "PROPOSE")
print(f"\n감사 로그 {len(wf.audit.log)}건, 해시 체인 검증: {'정상' if bad is None else f'seq {bad}에서 불일치'} -> 이력 무결성 {'100%' if bad is None else '훼손'}(목표 100%: {'달성' if bad is None else '미달'})")
saved = json.dumps(wf.audit.log[10], ensure_ascii=False)
wf.audit.log[10]["reason"] = "사후 수정"
print(f"10번째 기록 임의 수정 후 검증: {'위변조 탐지(seq ' + str(wf.audit.verify()) + ')' if wf.audit.verify() is not None else '탐지 실패'}")
wf.audit.log[10] = json.loads(saved)
print("승인된 건의 마지막 기록 예시:", {k: v for k, v in next((e for e in reversed(wf.audit.log) if e["event"] == "APPROVE" and e["case"].startswith("C")), {}).items() if k in ("seq", "who", "case", "reason", "evidence")})
print("※ 대체 부품의 규격 적합성 최종 판단과 조달 승인은 사람(기술검토관·보안담당·승인권자)이 하며, 시스템은 추천과 근거만 제공한다.")

plt.figure(figsize=(6, 3.5)); EN = {"승인": "approved", "반려": "rejected", "무응답 -> 보류": "on hold"}; plt.bar([EN.get(k, k) for k in st], list(st.values()), color="tab:blue"); plt.title("HITL workflow outcomes (12 cases)"); plt.ylabel("cases"); plt.tight_layout(); plt.show()

# ---------------------------------------------------------------------------
# [산출물 자동 생성] src/workflow/prc07_approval_system.py
# ---------------------------------------------------------------------------
m07_workflow_code = '''"""[PRC07_M07] 5단계 HITL 결재 FSM 및 SHA-256 해시 체인 감사 추적 모듈"""
import hashlib

class ApprovalWorkflowFSM:
    """PROPOSED -> SUBMITTED -> TECH_REVIEWED -> SEC_REVIEWED -> APPROVED"""
    def __init__(self):
        self.state = "PROPOSED"
'''
save_deliverable("src/workflow/prc07_approval_system.py", m07_workflow_code)


<details>
<summary><b>💡 [참고] 미션 07 예상 실행 결과 (Output)</b></summary>

```text
[결재 결과] 12건
  건     부품     후보     최종 상태                    비고
C01 P-0028 D-0046        반려              점수 미달 반려
C02 P-0056 D-0058        승인               승인권자 결정
C03 P-0087 D-0223        승인               승인권자 결정
C04 P-0036 D-0094 무응답 -> 보류 자동 확정 없음: 기술검토관 응답 대기
C05 P-0066 D-0172        승인               승인권자 결정
C06 P-0080 D-0205        승인               승인권자 결정
C07 P-0077 D-0049        반려              점수 미달 반려
C08 P-0045 D-0115        승인               승인권자 결정
C09 P-0003 D-0004        승인               승인권자 결정
C10 P-0062 D-0160        반려       보안 반려: 업체 확인 필요
C11 P-0053 D-0136        승인               승인권자 결정
C12 P-0012 D-0025        승인               승인권자 결정
상태 집계: {'승인': 8, '반려': 3, '무응답 -> 보류': 1} (모의 결재자 기준, 실제 결재는 사람이 한다)

[권한 우회·절차 위반 시도]
  시스템 자동 승인             -> 차단: 권한 없음(시스템 역할에 approve 권한 없음)
  조달담당이 규격 검토 대행        -> 차단: 권한 없음(조달담당 역할에 review_spec 권한 없음)
  검토 단계 건너뛰고 승인         -> 차단: 단계 위반(현재 상신, 필요 행위 review_spec)
  미등록 사용자 접근            -> 차단: 미등록 사용자
  사유 없는 검토              -> 차단: 사유·근거 입력 필수
  감사 로그 삭제              -> 차단: 감사 로그 삭제·수정 권한은 누구에게도 없음
  감사관이 승인 시도            -> 차단: 권한 없음(감사관 역할에 approve 권한 없음)
  종결된 건 재승인             -> 차단: 이미 종결된 건(반려)
  상신자가 본인 건 승인(분리 원칙)   -> 차단: 상신자와 승인자 분리 원칙 위반
우회·위반 시도 차단 9/9 (100%) (목표 100%: 달성)

감사 로그 71건, 해시 체인 검증: 정상 -> 이력 무결성 100%(목표 100%: 달성)
10번째 기록 임의 수정 후 검증: 위변조 탐지(seq 10)
승인된 건의 마지막 기록 예시: {'seq': 55, 'who': 'choi', 'case': 'C12', 'reason': '승인: 위험 부품 국산화 시험 착수', 'evidence': '후보 점수 0.84, 위험 62'}
※ 대체 부품의 규격 적합성 최종 판단과 조달 승인은 사람(기술검토관·보안담당·승인권자)이 하며, 시스템은 추천과 근거만 제공한다.
```
</details>
